# 7-Eleven App Event EDA Starter

이 노트북은 같은 폴더의 `app_event_eda_starter.py`를 실행해 앱 이벤트 데이터의 품질, 이벤트 퍼널, 채널/캠페인, 사용자 활동, 그래프 희소성 진단 산출물을 생성합니다.

- 입력: `data/processed/app_event_integrated/*.parquet`
- 출력: `eda/app_event_yumi/outputs/starter_eda/`
- 주의: `data/raw/`, `data/processed/`는 읽기 전용으로만 사용합니다.

In [1]:
from pathlib import Path
import runpy

candidates = [
    Path.cwd() / "app_event_eda_starter.py",
    Path.cwd() / "eda" / "app_event_yumi" / "app_event_eda_starter.py",
]
script = next((path for path in candidates if path.exists()), None)
if script is None:
    raise FileNotFoundError("app_event_eda_starter.py not found in app_event_yumi")

runpy.run_path(str(script), run_name="__main__")

Saved app event EDA outputs to: /Users/yumi/Seven-eleven/eda/app_event_yumi/outputs/starter_eda
shape: (1, 19)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ total_row ┆ parseable ┆ user_id_r ┆ appsflyer ┆ … ┆ af_conten ┆ pos_item_ ┆ item_name ┆ category │
│ s         ┆ _event_da ┆ ows       ┆ _id_rows  ┆   ┆ t_rows_pc ┆ code_rows ┆ _rows_pct ┆ _hierarc │
│ ---       ┆ te_rows   ┆ ---       ┆ ---       ┆   ┆ t         ┆ _pct      ┆ ---       ┆ hy_rows_ │
│ u32       ┆ ---       ┆ u32       ┆ u32       ┆   ┆ ---       ┆ ---       ┆ f64       ┆ pct      │
│           ┆ u32       ┆           ┆           ┆   ┆ f64       ┆ f64       ┆           ┆ ---      │
│           ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆ f64      │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ 29732312  ┆ 29732312  ┆ 29712894  ┆ 29732312  ┆ … ┆ 45.4089   ┆ 0.0       ┆ 0.0

{'__name__': '__main__',
 '__doc__': '\n7-Eleven app event EDA starter.\n\nBusiness goal\n-------------\nThis script gives the first reliable read of app behavior before modeling:\n\n1. Data quality\n   Check row counts, user-id coverage, product mapping coverage, and campaign\n   dimensions. Product fields are currently expected to be sparse or unmapped in\n   some integrated files, so product-level charts are generated only when mapped\n   names exist.\n\n2. Funnel and intent proxy\n   AppsFlyer event names are treated as behavioral states. Counts by event,\n   month, platform, media source, and campaign show where app traffic is coming\n   from and which interactions dominate.\n\n3. User and graph-readiness diagnostics\n   For HIN/GNN work, the relevant shape is a sparse user-content bipartite\n   interaction graph. We therefore estimate user activity distribution, content\n   node degree distribution, and top user-content edges using af_content as the\n   item/content proxy when PO

## 주요 확인 포인트

1. `data_quality_summary.csv`에서 상품 매핑률과 사용자 ID 커버리지를 확인합니다.
2. `event_name_counts.csv`로 앱 행동 퍼널의 상위 이벤트를 확인합니다.
3. `monthly_platform_counts.csv`, `media_source_counts.csv`, `top100_campaign_counts.csv`로 월/플랫폼/채널 차이를 봅니다.
4. `user_event_distribution_summary.csv`, `top50_users_by_event_count.csv`로 과활동 사용자 또는 비정상 ID 가능성을 점검합니다.
5. `graph_sparsity_summary.csv`, `top100_content_proxy_counts.csv`로 사용자-콘텐츠 그래프의 노드/엣지 희소성을 봅니다.



**1. 상품 EDA보다 “앱 행동 EDA”가 먼저다**

현재 `item_name`, `pos_item_code`, `category_hierarchy` 매핑률이 모두 `0%`입니다.  
즉, 지금 결과만으로는 “어떤 상품이 인기인가”를 상품명 기준으로 말하면 안 됩니다.

대신 `af_content`는 `45.4%` 존재하므로, 지금은 상품명이 아니라 **콘텐츠/이벤트 ID 기반 관심도 분석**까지만 가능합니다. 다음 단계에서 가장 먼저 해야 할 일은 `af_content`와 상품코드 매핑 복구입니다.

**2. 앱 사용은 로그인/이벤트/상품조회 중심이다**

총 이벤트 `29,732,312건` 중 상위 이벤트 비중은 다음과 같습니다.

- `af_login`: `27.9%`
- `af_content_view_eachevent`: `21.8%`
- `af_content_view_product`: `9.6%`
- `af_content_view_special`: `9.1%`
- `af_search_inventory`: `8.9%`
- `af_content_view_inventory`: `7.2%`

해석하면, 앱 이용의 큰 축은 **로그인 → 이벤트/특가/상품/재고 확인**입니다.  
일반적인 커머스 앱처럼 장바구니와 구매가 중심이라기보다, 편의점 앱 특성상 **이벤트 확인, 재고 확인, 픽업/예약 탐색**이 강하게 보입니다.

**3. 구매 전환은 아직 낮다**

대략적인 이벤트 퍼널로 보면:

- 상품조회 → 장바구니: 약 `17.2%`
- 장바구니 → 구매: 약 `24.1%`
- 상품조회 → 구매: 약 `4.1%`

즉 앱에서 상품을 보는 사람은 많지만, 실제 구매 이벤트까지 가는 비율은 낮습니다.  
이건 나쁜 신호라기보다, 세븐일레븐 앱이 “바로 결제”보다 **재고 확인/이벤트 확인/방문 유도형 앱**으로 쓰이고 있을 가능성이 큽니다.

**4. 재고 확인 기능은 굉장히 강하다**

`af_search_inventory`가 `2,646,992건`, `af_content_view_inventory`가 `2,146,290건`입니다.  
검색 재고 → 재고 상세 조회 비율이 약 `81.1%`라서, 재고 관련 행동은 꽤 목적성이 강합니다.

이건 신제품 예측 관점에서 중요합니다.  
앱에서 특정 상품의 재고 조회가 늘어난다면, 실제 POS 판매 전의 **선행 관심 신호**로 쓸 수 있습니다.

**5. 5월에 앱 이벤트가 크게 증가했다**

월별 이벤트 수:

- 2025-03: 약 `948만`
- 2025-04: 약 `826만`
- 2025-05: 약 `1,199만`

5월이 전체의 `40.3%`를 차지합니다. 특히 iOS 비중이 3월/4월보다 크게 올라갑니다.

- 3월 iOS 비중: `33.5%`
- 4월 iOS 비중: `31.6%`
- 5월 iOS 비중: `45.2%`

5월에 iOS 쪽 캠페인, 앱 업데이트, 이벤트, 데이터 수집 방식 변화가 있었는지 확인해야 합니다. 단순 수요 증가인지, 로깅 정책 변화인지 구분이 필요합니다.

**6. 채널 분석은 현재 한계가 크다**

`Media Source`가 `unknown`인 이벤트가 `93.6%`입니다.  
따라서 지금 상태로는 “어떤 광고 채널이 좋다”라고 강하게 말하기 어렵습니다.

확인 가능한 범위에서는:

- `googleadwords_int`: `5.76%`
- `nstation_int`: `0.44%`
- `kakao_int`: `0.11%`
- `l.point`: `0.11%`

즉 유입 채널 분석은 가능하지만, 대부분이 unknown이라 **attribution 정제 후 다시 봐야** 합니다.

**7. 시간대는 밤 10시~자정, 오전 10시가 강하다**

상위 시간대:

- 23시: `1,991,701건`
- 0시: `1,918,004건`
- 22시: `1,750,457건`
- 10시: `1,637,986건`
- 21시: `1,550,657건`

해석하면 앱 사용은 밤 시간대가 강합니다.  
편의점 소비 맥락상 야식, 재고 확인, 이벤트 확인, 픽업/예약 확인이 밤에 몰릴 가능성이 있습니다. 오전 10시도 높아서 이벤트 오픈/쿠폰/신상품 노출 시간과 관련이 있을 수 있습니다.

**8. 비정상 또는 대표 사용자 ID가 하나 있다**

`L00000000000` 사용자가 혼자 `3,209,311건`입니다.  
전체 이벤트의 약 `10.8%`를 한 ID가 차지합니다.

이 ID는 실제 개인 사용자라기보다 다음 중 하나일 가능성이 큽니다.

- 비로그인/익명 사용자 placeholder
- 테스트/시스템 계정
- user_id 누락 시 대체값
- 이벤트 로깅 오류

모델링이나 사용자 행동 분석에서는 이 ID를 제거하거나 별도 처리해야 합니다. 그대로 쓰면 사용자-상품 그래프가 심하게 왜곡됩니다.

**9. 그래프 모델 관점에서는 아직 “상품 그래프”가 아니라 “콘텐츠 그래프”다**

현재 `content_proxy` 노드는 `135개`, 사용자-콘텐츠 unique edge는 `1,607,887개`입니다.  
하지만 상품명 매핑이 없으므로 이건 상품 노드라기보다 앱 콘텐츠/이벤트/일부 content id 노드입니다.

따라서 HIN/GNN으로 연결하려면 다음 순서가 좋습니다.

1. `L00000000000` 같은 placeholder user 제거
2. `af_content` → 상품코드 매핑 복구
3. 이벤트 타입별 가중치 부여  
   예: 조회 < 재고조회 < 장바구니 < 구매
4. POS 판매 데이터와 날짜/상품코드 기준으로 결합
5. “앱 관심도 선행지표가 실제 판매를 설명하는가” 검증

요약하면, 지금 앱 데이터는 **신제품 관심 신호로 쓸 가능성이 높지만**, 바로 상품 예측에 넣기 전에는 `상품 매핑 복구`와 `비정상 사용자 제거`가 필수입니다.